In [ ]:
import ipaddress
import pandas as pd
import matplotlib.pyplot as plt

# Keep consistent with other analysis notebooks
TIME_OFFSET = 10800

In [ ]:
def plot_mean_modbus_response_time_per_dst_ip(
    input_csv,
    dst_ip,
    center_timestamp,
    interval_seconds,
    response_col="modbus.response_time",
    time_offset_seconds=TIME_OFFSET,
):
    """
    1) Plot mean(modbus.response_time) per dst IP in 1-second bins for
    [center_timestamp - interval_seconds, center_timestamp + interval_seconds].
    """
    ipaddress.ip_address(dst_ip)
    df = pd.read_csv(input_csv)

    required_cols = {"ip.dst", "frame.time_epoch", response_col}
    missing = required_cols - set(df.columns)
    if missing:
        raise ValueError(f"Missing required column(s): {sorted(missing)}")

    epoch = pd.to_numeric(df["frame.time_epoch"], errors="coerce")
    df = df.loc[epoch.notna()].copy()
    df["aligned_ts"] = pd.to_datetime(epoch[epoch.notna()] + float(time_offset_seconds), unit="s", errors="coerce")
    df[response_col] = pd.to_numeric(df[response_col], errors="coerce")

    center_ts = pd.to_datetime(center_timestamp, errors="raise")
    start_ts = center_ts - pd.Timedelta(seconds=int(interval_seconds))
    end_ts = center_ts + pd.Timedelta(seconds=int(interval_seconds))

    dst_df = df[df["ip.dst"].astype(str) == dst_ip].copy()
    window_df = dst_df[(dst_df["aligned_ts"] >= start_ts) & (dst_df["aligned_ts"] <= end_ts)].copy()
    window_df["second_bin"] = window_df["aligned_ts"].dt.floor("s")

    center_second = center_ts.floor("s")
    bins = pd.date_range(
        start=center_second - pd.Timedelta(seconds=int(interval_seconds)),
        end=center_second + pd.Timedelta(seconds=int(interval_seconds)),
        freq="s",
    )
    rel_x = (bins - center_second).total_seconds().astype(int)

    series = window_df.groupby("second_bin")[response_col].mean().reindex(bins)

    plt.figure(figsize=(11, 4))
    plt.plot(rel_x, series.values, marker="o")
    plt.axvline(0, color="black", linestyle="--", linewidth=1, label="Target timestamp")
    plt.xlabel("Seconds offset from target timestamp")
    plt.ylabel(f"mean({response_col})")
    plt.title(f"Mean modbus response time for dst={dst_ip} around {center_ts} (+/-{interval_seconds}s)")
    plt.grid(alpha=0.25)
    plt.legend()
    plt.tight_layout()
    plt.show()

    return series


def plot_max_modbus_response_time_per_dst_ip(
    input_csv,
    dst_ip,
    center_timestamp,
    interval_seconds,
    response_col="modbus.response_time",
    time_offset_seconds=TIME_OFFSET,
):
    """
    2) Plot max(modbus.response_time) per dst IP in 1-second bins.
    """
    ipaddress.ip_address(dst_ip)
    df = pd.read_csv(input_csv)

    required_cols = {"ip.dst", "frame.time_epoch", response_col}
    missing = required_cols - set(df.columns)
    if missing:
        raise ValueError(f"Missing required column(s): {sorted(missing)}")

    epoch = pd.to_numeric(df["frame.time_epoch"], errors="coerce")
    df = df.loc[epoch.notna()].copy()
    df["aligned_ts"] = pd.to_datetime(epoch[epoch.notna()] + float(time_offset_seconds), unit="s", errors="coerce")
    df[response_col] = pd.to_numeric(df[response_col], errors="coerce")

    center_ts = pd.to_datetime(center_timestamp, errors="raise")
    start_ts = center_ts - pd.Timedelta(seconds=int(interval_seconds))
    end_ts = center_ts + pd.Timedelta(seconds=int(interval_seconds))

    dst_df = df[df["ip.dst"].astype(str) == dst_ip].copy()
    window_df = dst_df[(dst_df["aligned_ts"] >= start_ts) & (dst_df["aligned_ts"] <= end_ts)].copy()
    window_df["second_bin"] = window_df["aligned_ts"].dt.floor("s")

    center_second = center_ts.floor("s")
    bins = pd.date_range(
        start=center_second - pd.Timedelta(seconds=int(interval_seconds)),
        end=center_second + pd.Timedelta(seconds=int(interval_seconds)),
        freq="s",
    )
    rel_x = (bins - center_second).total_seconds().astype(int)

    series = window_df.groupby("second_bin")[response_col].max().reindex(bins)

    plt.figure(figsize=(11, 4))
    plt.plot(rel_x, series.values, marker="o")
    plt.axvline(0, color="black", linestyle="--", linewidth=1, label="Target timestamp")
    plt.xlabel("Seconds offset from target timestamp")
    plt.ylabel(f"max({response_col})")
    plt.title(f"Max modbus response time for dst={dst_ip} around {center_ts} (+/-{interval_seconds}s)")
    plt.grid(alpha=0.25)
    plt.legend()
    plt.tight_layout()
    plt.show()

    return series


def plot_sum_retransmission_per_dst_ip(
    input_csv,
    dst_ip,
    center_timestamp,
    interval_seconds,
    retrans_col="tcp.analysis.retransmission",
    time_offset_seconds=TIME_OFFSET,
):
    """
    3) Plot sum(tcp.analysis.retransmission) per dst IP in 1-second bins.

    If retrans_col is non-numeric (e.g., text marker), non-empty values are treated as 1.
    """
    ipaddress.ip_address(dst_ip)
    df = pd.read_csv(input_csv)

    required_cols = {"ip.dst", "frame.time_epoch", retrans_col}
    missing = required_cols - set(df.columns)
    if missing:
        raise ValueError(f"Missing required column(s): {sorted(missing)}")

    epoch = pd.to_numeric(df["frame.time_epoch"], errors="coerce")
    df = df.loc[epoch.notna()].copy()
    df["aligned_ts"] = pd.to_datetime(epoch[epoch.notna()] + float(time_offset_seconds), unit="s", errors="coerce")

    numeric_vals = pd.to_numeric(df[retrans_col], errors="coerce")
    if numeric_vals.notna().any():
        df["retrans_numeric"] = numeric_vals.fillna(0)
    else:
        df["retrans_numeric"] = df[retrans_col].fillna("").astype(str).str.strip().ne("").astype(int)

    center_ts = pd.to_datetime(center_timestamp, errors="raise")
    start_ts = center_ts - pd.Timedelta(seconds=int(interval_seconds))
    end_ts = center_ts + pd.Timedelta(seconds=int(interval_seconds))

    dst_df = df[df["ip.dst"].astype(str) == dst_ip].copy()
    window_df = dst_df[(dst_df["aligned_ts"] >= start_ts) & (dst_df["aligned_ts"] <= end_ts)].copy()
    window_df["second_bin"] = window_df["aligned_ts"].dt.floor("s")

    center_second = center_ts.floor("s")
    bins = pd.date_range(
        start=center_second - pd.Timedelta(seconds=int(interval_seconds)),
        end=center_second + pd.Timedelta(seconds=int(interval_seconds)),
        freq="s",
    )
    rel_x = (bins - center_second).total_seconds().astype(int)

    series = window_df.groupby("second_bin")["retrans_numeric"].sum().reindex(bins, fill_value=0)

    plt.figure(figsize=(11, 4))
    plt.plot(rel_x, series.values, marker="o")
    plt.axvline(0, color="black", linestyle="--", linewidth=1, label="Target timestamp")
    plt.xlabel("Seconds offset from target timestamp")
    plt.ylabel(f"sum({retrans_col})")
    plt.title(f"Retransmission sum for dst={dst_ip} around {center_ts} (+/-{interval_seconds}s)")
    plt.grid(alpha=0.25)
    plt.legend()
    plt.tight_layout()
    plt.show()

    return series


def plot_packet_count_per_dst_ip(
    input_csv,
    dst_ip,
    center_timestamp,
    interval_seconds,
    time_offset_seconds=TIME_OFFSET,
):
    """
    4) Plot packet_count per dst IP in 1-second bins.
    """
    ipaddress.ip_address(dst_ip)
    df = pd.read_csv(input_csv)

    required_cols = {"ip.dst", "frame.time_epoch"}
    missing = required_cols - set(df.columns)
    if missing:
        raise ValueError(f"Missing required column(s): {sorted(missing)}")

    epoch = pd.to_numeric(df["frame.time_epoch"], errors="coerce")
    df = df.loc[epoch.notna()].copy()
    df["aligned_ts"] = pd.to_datetime(epoch[epoch.notna()] + float(time_offset_seconds), unit="s", errors="coerce")

    center_ts = pd.to_datetime(center_timestamp, errors="raise")
    start_ts = center_ts - pd.Timedelta(seconds=int(interval_seconds))
    end_ts = center_ts + pd.Timedelta(seconds=int(interval_seconds))

    dst_df = df[df["ip.dst"].astype(str) == dst_ip].copy()
    window_df = dst_df[(dst_df["aligned_ts"] >= start_ts) & (dst_df["aligned_ts"] <= end_ts)].copy()
    window_df["second_bin"] = window_df["aligned_ts"].dt.floor("s")

    center_second = center_ts.floor("s")
    bins = pd.date_range(
        start=center_second - pd.Timedelta(seconds=int(interval_seconds)),
        end=center_second + pd.Timedelta(seconds=int(interval_seconds)),
        freq="s",
    )
    rel_x = (bins - center_second).total_seconds().astype(int)

    series = window_df.groupby("second_bin").size().reindex(bins, fill_value=0)

    plt.figure(figsize=(11, 4))
    plt.plot(rel_x, series.values, marker="o")
    plt.axvline(0, color="black", linestyle="--", linewidth=1, label="Target timestamp")
    plt.xlabel("Seconds offset from target timestamp")
    plt.ylabel("packet_count")
    plt.title(f"Packet count for dst={dst_ip} around {center_ts} (+/-{interval_seconds}s)")
    plt.grid(alpha=0.25)
    plt.legend()
    plt.tight_layout()
    plt.show()

    return series


# Example usage:
# input_csv = "../train/ext_attack_nw.csv"
# dst_ip = "185.175.0.5"
# center_timestamp = "2023-03-19 03:01:57.813"
# x = 20
#
# plot_mean_modbus_response_time_per_dst_ip(input_csv, dst_ip, center_timestamp, x)
# plot_max_modbus_response_time_per_dst_ip(input_csv, dst_ip, center_timestamp, x)
# plot_sum_retransmission_per_dst_ip(input_csv, dst_ip, center_timestamp, x)
# plot_packet_count_per_dst_ip(input_csv, dst_ip, center_timestamp, x)